# XGBoost + SHAP 多因子选股策略

## 论文参考

- **标题**: Research on Quantitative Investment Strategies Based on Artificial Intelligence
- **作者**: Wenjie Sun
- **年份**: 2024
- **DOI**: 10.54254/2754-1169/70/2024MA0063

### 摘要

本文基于人工智能方法研究量化投资策略，采用 XGBoost 梯度提升模型进行股票截面收益预测，
并引入 SHAP (SHapley Additive exPlanations) 对模型进行可解释性分析。
策略在 A 股市场回测中取得了 **20.10%** 的收益率，展示了 AI 驱动选股的有效性与可解释性。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### XGBoost 目标函数

XGBoost 的正则化目标函数为：

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)$$

其中：
- $g_i = \partial_{\hat{y}^{(t-1)}} l(y_i, \hat{y}_i^{(t-1)})$ 为一阶梯度
- $h_i = \partial^2_{\hat{y}^{(t-1)}} l(y_i, \hat{y}_i^{(t-1)})$ 为二阶梯度 (Hessian)
- $\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$ 为正则化项

### 最优叶权重

对叶节点 $j$ 的最优权重为：
$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$

### 分裂增益

$$\text{Gain} = \frac{1}{2}\left[\frac{(\sum_{i \in I_L} g_i)^2}{\sum_{i \in I_L} h_i + \lambda} + \frac{(\sum_{i \in I_R} g_i)^2}{\sum_{i \in I_R} h_i + \lambda} - \frac{(\sum_{i \in I} g_i)^2}{\sum_{i \in I} h_i + \lambda}\right] - \gamma$$

### SHAP 值解释

SHAP 基于 Shapley 值理论，为每个特征分配贡献值：
$$\phi_j = \sum_{S \subseteq N \setminus \{j\}} \frac{|S|!(|N|-|S|-1)!}{|N|!} \left[f(S \cup \{j\}) - f(S)\right]$$

满足：$f(x) = \phi_0 + \sum_{j=1}^{M} \phi_j$，即所有特征的 SHAP 值之和等于预测值与基线之差。

In [ ]:
# ============================================================
# Cell 3: 动画展示 - 决策树特征空间划分
# ============================================================
import numpy as np
import plotly.graph_objects as go
from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)
N = 300
X1 = np.random.uniform(-3, 3, N)
X2 = np.random.uniform(-3, 3, N)
y_demo = np.sin(X1) * np.cos(X2) + np.random.normal(0, 0.2, N)
X_2d = np.column_stack([X1, X2])

# 生成网格用于可视化决策边界
xx1, xx2 = np.meshgrid(np.linspace(-3, 3, 60), np.linspace(-3, 3, 60))
grid = np.column_stack([xx1.ravel(), xx2.ravel()])

frames = []
for depth in range(1, 8):
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree.fit(X_2d, y_demo)
    pred_grid = tree.predict(grid).reshape(xx1.shape)
    mse = np.mean((y_demo - tree.predict(X_2d)) ** 2)

    frames.append(go.Frame(
        data=[
            go.Heatmap(x=np.linspace(-3, 3, 60), y=np.linspace(-3, 3, 60),
                       z=pred_grid, colorscale='RdBu_r', opacity=0.6,
                       showscale=True, colorbar=dict(title='预测值')),
            go.Scatter(x=X1, y=X2, mode='markers',
                       marker=dict(color=y_demo, colorscale='RdBu_r', size=4,
                                   line=dict(width=0.5, color='black')),
                       name='样本'),
        ],
        name=f'Depth={depth} (MSE={mse:.4f})'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='XGBoost: 决策树特征空间递归划分'),
        xaxis_title='特征 X1', yaxis_title='特征 X2',
        height=550, width=650,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='▶ 播放', method='animate',
                 args=[None, {'frame': {'duration': 800}, 'transition': {'duration': 400}}]),
            dict(label='⏸ 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0,
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import os
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_stock_daily, get_index_daily, get_multiple_stocks_daily
from shared.backtest_engine import MultiStockBacktester
from shared.factors import (
    momentum, volatility, rsi, macd, bollinger_bands, atr, adx, cci,
    turnover_factor, volume_price_corr, high_low_range, price_to_ma,
    winsorize, standardize
)
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_factor_importance, plot_returns_distribution
)
from shared.constants import DEFAULT_START, DEFAULT_END, CSI300_CODE

print('所有模块导入成功')
print(f'回测区间: {DEFAULT_START} - {DEFAULT_END}')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================

STOCK_POOL = [
    '600519', '601318', '600036', '000858', '601166',
    '600276', '601398', '600030', '000333', '002415',
    '600900', '601888', '000568', '002304', '601012',
    '600031', '603259', '600585', '000661', '601668',
]

try:
    stock_data = get_multiple_stocks_daily(STOCK_POOL, DEFAULT_START, DEFAULT_END)
    print(f'成功获取 {len(stock_data)} 只股票数据')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    stock_data = {}
    for sym in STOCK_POOL:
        np.random.seed(hash(sym) % 2**31)
        price = 50 + np.cumsum(np.random.randn(len(dates)) * 0.5)
        price = np.maximum(price, 5)
        stock_data[sym] = pd.DataFrame({
            'open': price * (1 + np.random.randn(len(dates)) * 0.005),
            'close': price,
            'high': price * (1 + np.abs(np.random.randn(len(dates)) * 0.01)),
            'low': price * (1 - np.abs(np.random.randn(len(dates)) * 0.01)),
            'volume': np.random.randint(100000, 5000000, len(dates)).astype(float),
            'turnover': np.random.uniform(0.3, 5.0, len(dates)),
        }, index=dates)

try:
    benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)
    benchmark_prices = benchmark['close']
    print(f'基准数据: {len(benchmark_prices)} 条')
except Exception as e:
    print(f'基准获取异常: {e}')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    benchmark_prices = pd.Series(5000 + np.cumsum(np.random.randn(len(dates)) * 10), index=dates)

In [ ]:
# ============================================================
# Cell 6: 因子工程 (含偏度、峰度)
# ============================================================

def build_features_xgb(df):
    """为单只股票构建因子, 增加偏度和峰度"""
    features = pd.DataFrame(index=df.index)
    ret = df['close'].pct_change()

    # 动量因子
    mom = momentum(df['close'], periods=[5, 10, 20])
    features = pd.concat([features, mom], axis=1)

    # 波动率
    vol = volatility(df['close'], windows=[10, 20])
    features = pd.concat([features, vol], axis=1)

    # RSI, MACD, Bollinger
    features['rsi_14'] = rsi(df['close'], 14)
    macd_df = macd(df['close'])
    features['macd_hist'] = macd_df['histogram']
    bb = bollinger_bands(df['close'], 20)
    features['bb_pctb'] = bb['bb_pctb']
    features['bb_width'] = bb['bb_width']

    # 换手率
    if 'turnover' in df.columns:
        features['turnover'] = df['turnover']
        features['turnover_ma5'] = df['turnover'].rolling(5).mean()

    # 量价相关
    if 'volume' in df.columns:
        features['vp_corr_20'] = volume_price_corr(df['close'], df['volume'], 20)

    # 价格偏离均线
    ptm = price_to_ma(df['close'], windows=[5, 10, 20])
    features = pd.concat([features, ptm], axis=1)

    # ATR
    if all(c in df.columns for c in ['high', 'low']):
        features['atr_14'] = atr(df['high'], df['low'], df['close'], 14)
        features['hl_range'] = high_low_range(df['high'], df['low'], df['close'])

    # >>> 新增: 偏度和峰度 <<<
    features['skew_20'] = ret.rolling(20).skew()
    features['kurt_20'] = ret.rolling(20).kurt()
    features['skew_60'] = ret.rolling(60).skew()
    features['kurt_60'] = ret.rolling(60).kurt()

    return features


all_features = []
for sym, df in stock_data.items():
    if len(df) < 60:
        continue
    feats = build_features_xgb(df)
    feats['fwd_return_20d'] = df['close'].pct_change(20).shift(-20)
    feats['symbol'] = sym
    all_features.append(feats)

panel = pd.concat(all_features).reset_index()
if 'index' in panel.columns:
    panel.rename(columns={'index': 'date'}, inplace=True)

FEATURE_COLS = [c for c in panel.columns if c not in ['date', 'symbol', 'fwd_return_20d']]
print(f'因子面板: {panel.shape[0]} 行 x {len(FEATURE_COLS)} 个因子')
print(f'因子列表: {FEATURE_COLS}')

In [ ]:
# ============================================================
# Cell 7: XGBoost 模型训练 + SHAP 分析 (滚动窗口)
# ============================================================

TRAIN_MONTHS = 12
TOP_K = 10

panel['date'] = pd.to_datetime(panel['date'])
panel['year_month'] = panel['date'].dt.to_period('M')
months = sorted(panel['year_month'].unique())
print(f'数据覆盖 {len(months)} 个月: {months[0]} ~ {months[-1]}')

xgb_params = {
    'objective': 'reg:squarederror',
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'seed': 42,
    'verbosity': 0,
}

selections = {}
all_importances = []
shap_values_list = []
last_model = None
last_X_test = None

for i in range(TRAIN_MONTHS, len(months) - 1):
    train_months_range = months[i - TRAIN_MONTHS: i]
    test_month = months[i]

    train_data = panel[panel['year_month'].isin(train_months_range)].copy()
    test_data = panel[panel['year_month'] == test_month].copy()

    train_data = train_data.dropna(subset=FEATURE_COLS + ['fwd_return_20d'])
    test_data_clean = test_data.dropna(subset=FEATURE_COLS)

    if len(train_data) < 50 or len(test_data_clean) < 5:
        continue

    X_train = train_data[FEATURE_COLS].values
    y_train = train_data['fwd_return_20d'].values
    X_test = test_data_clean[FEATURE_COLS].values

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURE_COLS)
    dtest = xgb.DMatrix(X_test, feature_names=FEATURE_COLS)

    model = xgb.train(xgb_params, dtrain, num_boost_round=200)

    pred = model.predict(dtest)
    test_data_clean = test_data_clean.copy()
    test_data_clean['pred_return'] = pred

    last_day = test_data_clean.groupby('symbol')['pred_return'].last()
    top_stocks = last_day.nlargest(TOP_K).index.tolist()

    rebal_date = test_data_clean['date'].max()
    selections[rebal_date] = top_stocks

    imp = model.get_score(importance_type='gain')
    all_importances.append(imp)

    last_model = model
    last_X_test = X_test

    if (i - TRAIN_MONTHS) % 6 == 0:
        print(f'  {test_month}: 选股 {top_stocks[:5]}...')

print(f'\n共生成 {len(selections)} 期选股结果')

# 平均重要性
avg_importance = {}
for col in FEATURE_COLS:
    vals = [imp.get(col, 0) for imp in all_importances]
    avg_importance[col] = np.mean(vals)

# SHAP 分析 (最后一期)
print('\n计算 SHAP 值 (最后一期模型)...')
explainer = shap.TreeExplainer(last_model)
shap_vals = explainer.shap_values(last_X_test)
print(f'SHAP 值矩阵: {shap_vals.shape}')

In [ ]:
# ============================================================
# Cell 8: 信号生成与回测
# ============================================================

all_prices = {sym: df['close'] for sym, df in stock_data.items()}

backtester = MultiStockBacktester(initial_capital=1_000_000, rebalance_freq='M')
result = backtester.run(
    all_prices=all_prices,
    selections=selections,
    benchmark_prices=benchmark_prices
)

print('=== XGBoost + SHAP 选股策略回测结果 ===')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化 + SHAP 图
# ============================================================

# 1. 收益曲线
benchmark_equity = None
if result.get('benchmark_returns') is not None:
    benchmark_equity = (1 + result['benchmark_returns']).cumprod() * result['equity_curve'].iloc[0]
plot_equity_curve(result['equity_curve'], benchmark_equity,
                  title='XGBoost + SHAP 选股 - 累计收益')

# 2. 回撤图
plot_drawdown(result['equity_curve'], title='XGBoost + SHAP 选股 - 回撤')

# 3. 因子重要性 (XGBoost gain)
plot_factor_importance(avg_importance, title='XGBoost 因子重要性 (平均 Gain)', top_n=15)

# 4. SHAP summary plot
print('SHAP Summary Plot:')
shap.summary_plot(shap_vals, features=last_X_test, feature_names=FEATURE_COLS,
                  max_display=15, show=True)

# 5. SHAP bar plot
print('SHAP 平均绝对贡献:')
shap.summary_plot(shap_vals, features=last_X_test, feature_names=FEATURE_COLS,
                  plot_type='bar', max_display=15, show=True)

# 6. 收益率分布
plot_returns_distribution(result['returns'], title='XGBoost 策略日收益率分布')

# 7. 绩效表
plot_metrics_table(result['metrics'], title='XGBoost + SHAP 选股策略绩效指标')

## 结果讨论

### XGBoost vs LightGBM

1. **XGBoost 使用二阶梯度信息** (Hessian)，理论上在凸损失函数下有更精确的近似
2. **训练速度**: LightGBM 通常更快 (histogram + GOSS)，XGBoost 使用精确贪心或 histogram 方法
3. 两者在实际选股效果上差异不大，更多取决于超参数调优

### SHAP 可解释性

- SHAP summary plot 直观展示了每个因子对预测的全局贡献
- 偏度 (skew) 和峰度 (kurt) 因子可能捕捉到尾部风险信息
- SHAP 值满足加性分解，可以精确追踪每个预测值的来源

### 改进方向

- 使用 SHAP interaction values 分析因子交互效应
- 基于 SHAP 值进行因子筛选 (去除贡献小的因子)
- 引入基本面因子和行业哑变量
- 尝试 XGBoost 的 `rank:pairwise` 目标函数